# Quishing Reproduction and Artifact Verification
## Q14_04_QA01 — Verify the Final R2 System Outside the Research Notebook

This notebook exports the already fitted **hostname-and-unreserved-path normalized** models and recomputes their predictions. It checks that results remain consistent with the saved research evidence in a separate Python process, then in a newly created virtual environment.

No model is trained, no threshold is recalibrated and no destination is contacted. The three EXP08 seeds remain available; seed 42 is used for the transformation study by a fixed presentation convention, not by performance selection.

<font color="violet">QA passes only after the verification cells run successfully on Cube-AI. A prepared export is not a verified package. The final publication decision remains separate from technical reproduction.</font>

Unzip the supplied QA folder into the experiment directory. Keep this notebook beside `qa_support/`. The existing `outputs/quishing02` tree must remain accessible. Open a **fresh notebook kernel in the ai-sec-research environment** and execute the cells in order.

### What We Verify

| Step | Purpose | Evidence produced |
|---|---|---|
| 1 | Identify the exact saved runs and check their hashes | Source artifact audit |
| 2 | Bind the checkpoint functions to an importable module | Frozen models, thresholds and source lineage |
| 3 | Assemble verification fixtures | 1,956 clean PNGs, 69 known attack PNGs and 755 URL pairs |
| 4 | Recalculate in a separate process | Saved-score parity for all seeds; full R2 transformation results for seed 42 |
| 5 | Install exact dependency versions in a new virtual environment | Installation log; no inherited system site-packages |
| 6 | Verify fresh-environment parity | All clean and known attack PNGs; 16 samples per transformed condition |
| 7 | Record conclusions and publication inputs | QA report and artifact inventory |

The eight transformations are the three right-angle rotations, JPEG quality 90/60, blur sigma 0.5/1.0 and contrast factor 0.65. The full transformation replay uses all 1,956 clean examples in the first verification process. Fresh-environment parity uses a fixed 16-image subset for each transformation.

In [4]:
from pathlib import Path
from zipfile import ZipFile

with ZipFile("04_quishing_reproduction_qa_bundle.zip") as archive:
    for name in archive.namelist():
        if name.startswith("qa_support/"):
            archive.extract(name, Path.cwd())

print("QA helpers available:", (Path.cwd() / "qa_support/qa_tools.py").is_file())

QA helpers available: True


In [5]:
# Load the reviewable export and verification helpers supplied with this notebook.
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

SUPPORT_DIR = Path.cwd() / "qa_support"
if not (SUPPORT_DIR / "qa_tools.py").is_file():
    raise FileNotFoundError("Unzip the QA bundle here so qa_support/ is beside this notebook.")
sys.path.insert(0, str(SUPPORT_DIR))
import qa_tools as qa

print("QA helpers loaded. No model, threshold or research result has been changed.")

QA helpers loaded. No model, threshold or research result has been changed.


### 1. Select the Saved Research Runs

We use the EXP08 run already exercised by the technical demonstration. We reuse the previously captured decoder source, whose behaviour will be checked again against the final R2 models.

If more than one matching run exists, the notebook stops and lists the candidates. Choose the intended directory explicitly rather than silently taking the newest result.

In [6]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "outputs/quishing02").is_dir()), Path.cwd())
EXP08_OVERRIDE = None
PATH_SEARCH_OVERRIDE = None
CORRECTED_OVERRIDE = None
DECODER_PACKAGE_OVERRIDE = None

paths = qa.discover(PROJECT_ROOT, exp08=EXP08_OVERRIDE,
    search=PATH_SEARCH_OVERRIDE, corrected=CORRECTED_OVERRIDE,
    decoder=DECODER_PACKAGE_OVERRIDE)
display(pd.DataFrame([{"artifact": name, "directory": str(path)}
                      for name, path in paths.items()]))

,artifact,directory
0,root,/mnt/nvme/experiments/AI-Security/Experiment 1...
1,exp08,/mnt/nvme/experiments/AI-Security/Experiment 1...
2,search,/mnt/nvme/experiments/AI-Security/Experiment 1...
3,corrected,/mnt/nvme/experiments/AI-Security/Experiment 1...
4,decoder,/mnt/nvme/experiments/AI-Security/Experiment 1...


#### Output Explanation

The selected EXP08 run contains the three fitted checkpoints and their calibrated thresholds. The path-search run supplies the known attacks and original clean PNGs. The earlier decoder package supplies source code, not the earlier classifier.

A directory mismatch must be resolved before exporting. These checks use existing files; they do not execute notebook `02` or require its live kernel.

In [7]:
source_audit = qa.verify_sources(paths)
display(source_audit)
print("Saved evidence, all three model checkpoints and the captured decoder source passed integrity checks.")

,file,sha256,checked
0,outputs/quishing02/frozen_candidates/character...,37374741de88e5b5328ea2573f556858ff7a141f7c1a79...,True
1,outputs/quishing02/frozen_candidates/character...,50459b7319dd80d0e7fb48296ec4931346e02ad9057bcb...,True
2,outputs/quishing02/frozen_candidates/character...,23ad0e08067c9c8e5310b1eb4aa388a79f38c9cef4c8a3...,True
3,outputs/quishing02/frozen_candidates/character...,54305188e786363b63ebe1420bb7e5a575e23fb76b9901...,True
4,outputs/quishing02/frozen_candidates/character...,92414c41584d75288c4eb575fecfea22311e26c1214660...,True
5,outputs/quishing02/frozen_candidates/character...,d94d241b69cd69c44f63fe73500560e98d1e8389bb4c94...,True
6,outputs/quishing02/frozen_candidates/character...,d45af3ea257f5ae674033a488b170a6f2bfe0a70c48c2d...,True
7,outputs/quishing02/frozen_candidates/character...,a396b440dd236069cf0790e3aacc455e42376a3ce5d650...,True
8,outputs/quishing02/frozen_candidates/character...,2c57f306b8be52a6b601645c5e1c152235ec71ef09ad59...,True
9,outputs/quishing02/frozen_candidates/character...,2c3c7279eefc78d0918feafe686e8bfd70b5e068d6184d...,True


Saved evidence, all three model checkpoints and the captured decoder source passed integrity checks.


### 2. Export the Frozen R2 Models

The original joblib checkpoints reference Python functions formerly defined in the research kernel. We bind those references to the supplied `quishing_features.py`, which contains the seven exact feature/normalization functions extracted from the research notebook.

The exporter temporarily provides those reviewed functions to resolve our own trusted checkpoints. It then saves models that reference an ordinary importable module. Learned-state fingerprints and original checkpoint hashes are recorded; a changed learned-state fingerprint stops the export.

<font color="violet">The export changes where functions are imported from. It does not fit a classifier or select a new threshold. Loading serialized models requires a trusted source; matching hashes alone do not establish trust.</font>

In [8]:
qa_context = qa.create_package(paths)
display(qa_context["lineage"][["seed", "source_checkpoint", "threshold", "learned_state_fingerprint"]])
print(f"Prepared package: {qa_context['package']}\nVerification is still pending.")

,seed,source_checkpoint,threshold,learned_state_fingerprint
0,42,outputs/quishing02/frozen_candidates/character...,0.533642,4629e22db7473321b2cd5a68b70a59cd
1,142,outputs/quishing02/frozen_candidates/character...,0.533642,4629e22db7473321b2cd5a68b70a59cd
2,242,outputs/quishing02/frozen_candidates/character...,0.533642,4629e22db7473321b2cd5a68b70a59cd


Prepared package: <experiment-root>/outputs/quishing04/qa_20261122T140752771854Z/package
Verification is still pending.


#### Output Explanation

The newly created `outputs/quishing04/qa_.../package` directory contains three models, their exact thresholds, importable feature and decoder modules, an inference API and pinned requirements.

The original research artifacts remain unchanged. Numerical parity is checked next; successful serialization alone does not establish reproduction.

### 3. Prepare Traceable Verification Inputs

We copy the actual clean and attacked PNGs into the verification package. The clean text reference is represented by its saved SHA-256 fingerprint. Only the text recovered by the decoder reaches the classifier.

For attacked PNGs, we verify the exact candidate text and compare the new score with the original-source score under the tested normalization-invariance condition. This is explicitly an invariance check, not a claim that a historical per-candidate R2 numerical score existed in the demonstration files.

<font color="violet">Verification fixtures contain historical dataset content, including phishing addresses. They stay local during QA. The final inventory distinguishes these fixtures for review before GitHub publication.</font>

In [9]:
fixture_summary = qa.prepare_inputs(qa_context)
display(fixture_summary)
print("Verification inputs copied and hashed. No URL was visited and no new attack search was run.")

,group,sources
0,clean QR,1956
1,stored known path attacks,69
2,retained URL pairs,755
3,transformations on final seed 42,8 x 1956 in baseline; 8 x 16 in fresh parity


Verification inputs copied and hashed. No URL was visited and no new attack search was run.


### 4. Recalculate in a Separate Python Process

This process imports the exported modules and models without notebook globals. It checks:

- All 1,956 clean payloads are recovered exactly; scores and decisions match saved EXP08 results for each seed.
- All 69 stored known path attacks are recovered exactly and detected for each seed.
- All 755 retained URL pairs have identical original/variant scores for each seed.
- The eight transformations are replayed on all 1,956 examples with the final seed-42 model.

Exact recovered payloads must reproduce their clean scores. Decode failures, changed accepted payloads and abstentions remain visible in the transformation results; we do not force them to match historical behaviour from an older revision.

In [10]:
same_environment_summary = qa.verify_same_environment(qa_context)
display(same_environment_summary.round(6))
print("Separate-process verification passed in the installed environment. Fresh-environment verification remains.")

Clean QR: 250/1956
Clean QR: 500/1956
Clean QR: 750/1956
Clean QR: 1000/1956
Clean QR: 1250/1956
Clean QR: 1500/1956
Clean QR: 1750/1956
Clean QR: 1956/1956
rotate_90: 500/1956
rotate_90: 1000/1956
rotate_90: 1500/1956
rotate_90: 1956/1956
rotate_180: 500/1956
rotate_180: 1000/1956
rotate_180: 1500/1956
rotate_180: 1956/1956
rotate_270: 500/1956
rotate_270: 1000/1956
rotate_270: 1500/1956
rotate_270: 1956/1956
jpeg_90: 500/1956
jpeg_90: 1000/1956
jpeg_90: 1500/1956
jpeg_90: 1956/1956
jpeg_60: 500/1956
jpeg_60: 1000/1956
jpeg_60: 1500/1956
jpeg_60: 1956/1956
blur_0.5: 500/1956
blur_0.5: 1000/1956
blur_0.5: 1500/1956
blur_0.5: 1956/1956
blur_1.0: 500/1956
blur_1.0: 1000/1956
blur_1.0: 1500/1956
blur_1.0: 1956/1956
contrast_0.65: 500/1956
contrast_0.65: 1000/1956
contrast_0.65: 1500/1956
contrast_0.65: 1956/1956
Verification passed; results saved to same_environment_results.json


,group,seed,QR_images,accepted,phishing_detected,false_alarms,recall_all,FPR_all,abstentions
0,clean,42,1956,1956,870,28,0.895062,0.028455,0
1,clean,142,1956,1956,870,28,0.895062,0.028455,0
2,clean,242,1956,1956,870,28,0.895062,0.028455,0
3,stored_path_attack,42,69,69,69,0,1.000000,NaN,0
4,stored_path_attack,142,69,69,69,0,1.000000,NaN,0
5,stored_path_attack,242,69,69,69,0,1.000000,NaN,0
6,rotate_90,42,1956,1956,870,28,0.895062,0.028455,0
7,rotate_180,42,1956,1956,870,28,0.895062,0.028455,0
8,rotate_270,42,1956,1956,870,28,0.895062,0.028455,0
9,jpeg_90,42,1956,1956,870,28,0.895062,0.028455,0


Separate-process verification passed in the installed environment. Fresh-environment verification remains.


#### Output Explanation

This cell performs **new classifier inference** using the exported fitted models. It differs from the demonstration notebook, which displayed saved scores.

A lower recall under a transformation may expose an availability failure. In particular, zero false alarms with zero accepted decodes must not be interpreted as robust detection. The transformation table reports the final revision's measured behaviour, even when that behaviour is unfavourable.

### 5. Create a Fresh Isolated Environment

We create a new virtual environment using this interpreter, with `system_site_packages=False`, and install the recorded dependency versions into that environment only. Package installation may require access to the configured Python package index.

The cell preserves an installation log. If a pinned version is unavailable or a dependency cannot load, QA stops: the notebook does not fall back to the existing environment or silently substitute another version.

<font color="violet">This checks an isolated Python environment on the same host and Python major/minor version. It does not claim portability to every operating system or a clean machine.</font>

In [11]:
fresh_environment_summary = qa.install_fresh_environment(qa_context)
display(fresh_environment_summary)
print("Fresh environment installed without inherited system site-packages.")

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached zxing_cpp-3.1.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.1 MB)
Using cached sc

,environment,inherits_system_packages,requirements,install
0,/mnt/nvme/experiments/AI-Security/Experiment 1...,False,exact recorded source versions,passed


Fresh environment installed without inherited system site-packages.


### 6. Verify Fresh-Environment Inference

The new interpreter loads the package and verifies its hashes. It then recomputes all 1,956 clean QR examples and all 69 known attack PNGs with all three models, repeats all 755 retained URL pairs, and checks 16 fixed examples per transformation with seed 42.

The same score tolerances apply: relative tolerance `1e-10` and absolute tolerance `1e-12`. Acceptance, payload identity and decisions must match exactly.

In [12]:
fresh_environment_results = qa.verify_fresh_environment(qa_context)
display(fresh_environment_results.round(6))
print("Fresh isolated-environment inference passed. The final cell compares both process outputs.")

Clean QR: 250/1956
Clean QR: 500/1956
Clean QR: 750/1956
Clean QR: 1000/1956
Clean QR: 1250/1956
Clean QR: 1500/1956
Clean QR: 1750/1956
Clean QR: 1956/1956
rotate_90: 16/16
rotate_180: 16/16
rotate_270: 16/16
jpeg_90: 16/16
jpeg_60: 16/16
blur_0.5: 16/16
blur_1.0: 16/16
contrast_0.65: 16/16
Verification passed; results saved to fresh_environment_results.json


,group,seed,QR_images,accepted,phishing_detected,false_alarms,recall_all,FPR_all,abstentions
0,clean,42,1956,1956,870,28,0.895062,0.028455,0
1,clean,142,1956,1956,870,28,0.895062,0.028455,0
2,clean,242,1956,1956,870,28,0.895062,0.028455,0
3,stored_path_attack,42,69,69,69,0,1.000000,NaN,0
4,stored_path_attack,142,69,69,69,0,1.000000,NaN,0
5,stored_path_attack,242,69,69,69,0,1.000000,NaN,0
6,rotate_90,42,16,16,10,0,1.000000,0.000000,0
7,rotate_180,42,16,16,10,0,1.000000,0.000000,0
8,rotate_270,42,16,16,10,0,1.000000,0.000000,0
9,jpeg_90,42,16,16,10,0,1.000000,0.000000,0


Fresh isolated-environment inference passed. The final cell compares both process outputs.


#### Output Explanation

The fresh environment confirms that exported inference does not depend on a research notebook kernel. The complete transformed cohort was evaluated in step 4; step 6 checks fresh-environment parity on a documented subset of those transformations.

These are reproduction and known-attack regression checks. No training, recalibration, independent held-out evaluation or renewed adaptive attack search occurs.

### 7. Record the QA Decision and Publication Inputs

We compare both process results by transformation, source ID and training seed. We require identical decoded-payload hashes, acceptance decisions and classifier decisions, with matching scores within the declared tolerance. Clean detection and false-alarm totals must also match the saved research results.

The QA report distinguishes **technical inference reproduction** from publication readiness. A file inventory records which packaged files contain historical dataset fixtures. Review those inputs and integrate the final QA scope into the report and GitHub README before publication.

In [13]:
qa_decision = qa.finalize(qa_context)
display(qa_decision)
print(f"QA report: {qa_context['run_dir'] / 'qa_report.json'}"
      f"\nPublication input inventory: {qa_context['run_dir'] / 'publication_input_inventory.csv'}")

,check,status
0,same-environment inference and saved-score parity,passed
1,fresh isolated-environment inference and parity,passed
2,model and threshold preservation,passed
3,publication of dataset fixtures and final docu...,review pending


QA report: <experiment-root>/outputs/quishing04/qa_20261122T140752771854Z/qa_report.json
Publication input inventory: <experiment-root>/outputs/quishing04/qa_20261122T140752771854Z/publication_input_inventory.csv


### Inspect Newly Calculated Demonstration Results

The selected examples from the demonstration have source IDs 31 (legitimate), 5 (phishing) and 4818 (stored path attack). We display their newly calculated seed-42 results from the successful fresh-environment run.

The values below come from this QA execution, not the old saved demonstration scores. After verifying them, the demonstration can be connected to `quishing_pipeline.analyse_png` for direct inference.

In [14]:
fresh_rows = pd.DataFrame(qa_context["fresh_result"]["records"])
demo_rows = fresh_rows.loc[fresh_rows["seed"].eq(42) & (
    (fresh_rows["group"].eq("clean") & fresh_rows["image_index"].isin([31, 5])) |
    (fresh_rows["group"].eq("stored_path_attack") & fresh_rows["image_index"].eq(4818)))]
display(demo_rows[["group", "image_index", "accepted", "score", "prediction"]].round(9))
print("Freshly recomputed classifier results; model weights and thresholds were preserved.")

,group,image_index,accepted,score,prediction
0,clean,5,True,0.989164,1
3,clean,31,True,0.036433,0
5958,stored_path_attack,4818,True,0.996262,1


Freshly recomputed classifier results; model weights and thresholds were preserved.


## QA Conclusion

Interpret this conclusion only after all preceding verification cells pass. The QA report records the actual outcome and scope of this run.

Successful completion establishes that the exported frozen R2 models reproduce the recorded clean scores and decisions, detect the known retained path-attack PNGs, and operate outside the research notebook in an isolated same-host environment. It also supplies a complete final-revision transformation replay and a documented fresh-environment parity subset.

<font color="violet">These checks do not remove development-selection bias, establish general attack robustness, rerun the complete training study, or authorize publication of historical dataset fixtures. The 69 × 69 input convention and the path-equivalence conditions remain part of the result.</font>

**README handoff:** distinguish saved-score demonstration from the fresh inference performed here; identify R1 versus R2; document selected examples, dependencies, fixture availability, abstention, residual errors and the exact reproduction scope.